# Data Collection Workflow

### Required libraries

In [1]:
!python.exe -m pip install -U pip
!pip install beautifulsoup4 pandas selenium requests

In [2]:
import requests, re, json, time
from bs4 import BeautifulSoup
import pandas as pd
from requests.adapters import HTTPAdapter
from requests.packages.urllib3.util.retry import Retry
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, ElementNotInteractableException
from bs4 import BeautifulSoup

In [3]:
# Configure session with retry strategy
def create_session():
    session = requests.Session()
    retry_strategy = Retry(
        total=3,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["HEAD", "GET", "OPTIONS"],  # Updated from method_whitelist
        backoff_factor=1
    )
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    return session

## Load existing data

In [4]:
dataset_dataframe = pd.read_csv("../data/1_boxers_dataset.csv")
dataset_dataframe

,Name,Height (ft),Reach (cm),Reach (inches),Stance,Total fights,Wins,Losses,Draws,Date_Birth,Height (cm),No contests,Age,Height
0,Andy Ruiz Jr.,6 ft 2 in,188.0,74.0,Orthodox,38.0,35.0,2.0,1.0,NaN,NaN,NaN,NaN,NaN
1,Dillian Whyte,6 ft 4 in,198.0,78.0,Orthodox,NaN,1.0,0.0,NaN,11 April 1988,NaN,NaN,NaN,NaN
2,Alan Picasso Romero,5 ft 8 in,178.0,70.0,Orthodox,32.0,31.0,0.0,1.0,NaN,NaN,NaN,NaN,NaN
3,Gary Antuanne Russell,5 ft 10 in,175.0,69.0,Southpaw,19.0,18.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN
4,Gilberto Ramirez,NaN,191.0,75.0,Southpaw,49.0,48.0,1.0,NaN,19 June 1991,189.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76,Christian Mbilli,NaN,183.0,72.0,Orthodox,29.0,29.0,NaN,NaN,26 April 1995,174.0,NaN,NaN,NaN
77,Arnold Barboza Jr.,5 ft 9 in,183.0,72.0,Orthodox,33.0,32.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN
78,Elwin Soto,5 ft 3 in,NaN,NaN,Orthodox,24.0,21.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN
79,Sandor Martín,5 ft 7 in,175.0,69.0,Southpaw,46.0,42.0,4.0,NaN,22 August 1993,NaN,NaN,NaN,NaN


## Retrieval data on [BoxRec](/data/active_boxing_fighters.txt)

In [5]:
# Configuration des identifiants BoxRec
global BOXREC_USERNAME, BOXREC_PASSWORD
BOXREC_USERNAME, BOXREC_PASSWORD = "charifmcm@gmail.com", "Test2025!"  # Remplacez par vos identifiants


def create_selenium_driver(show_browser=True):
    options = Options()
    
    # Only add headless if explicitly requested AND show_browser is False
    if not show_browser:
        options.add_argument("--headless")
    
    # Remove problematic arguments that might prevent browser from opening
    # options.add_argument("--disable-gpu")  # Commented out
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")  # Added for stability
    
    # Keep these for bot detection avoidance but don't prevent browser opening
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--disable-extensions")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option('useAutomationExtension', False)
    
    # Add user agent to appear more like a real browser
    options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    
    # Ensure window opens with reasonable size
    if show_browser:
        options.add_argument("--window-size=1920,1080")
        options.add_argument("--start-maximized")
    
    try:
        print("Creating Chrome driver...")
        driver = webdriver.Chrome(options=options)
        driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
        
        # Set reasonable timeouts
        driver.implicitly_wait(10)
        driver.set_page_load_timeout(30)
        
        print("Chrome driver created successfully - browser window should be visible")
        return driver
    except Exception as e:
        print(f"Error creating Chrome driver: {e}")
        print("Make sure Chrome is installed and chromedriver is in your PATH")
        return None

def click_closing_links(driver):
    """Click on all 'a' tags with id containing 'closingLink' to close overlays"""
    try:
        closing_links = driver.find_elements(By.XPATH, "//a[contains(@id, 'closingLink')]")
        if not closing_links:
            print("No closing links found")
            return
        
        print(f"Found {len(closing_links)} closing links")
        
        # Click links one by one, re-finding elements each time to avoid stale references
        for i in range(len(closing_links)):
            try:
                # Re-find the closing links each time to avoid stale element references
                current_links = driver.find_elements(By.XPATH, "//a[contains(@id, 'closingLink')]")
                if i < len(current_links):
                    link = current_links[i]
                    # Check if element is still clickable
                    if link.is_displayed() and link.is_enabled():
                        href = link.get_attribute('href')
                        link.click()
                        print(f"Clicked on closing link {i+1}: {href}")
                        time.sleep(1)  # Small delay to allow overlay to close
                    else:
                        print(f"Closing link {i+1} is not clickable")
                else:
                    print(f"Closing link {i+1} no longer exists")
            except Exception as e:
                print(f"Error clicking closing link {i+1}: {str(e)[:100]}...")  # Truncate long error messages
                continue  # Continue with next link
    except Exception as e:
        print(f"Error finding closing links: {e}")

# Login function: Connect to BoxRec
def login_boxrec(driver,
                 username, password):
    """Login to BoxRec to access full data"""
    try:
        print("Attempting to login to BoxRec...")
        driver.get("https://boxrec.com/en/login")
        time.sleep(3)  # Wait for page to load

        # Handle potential overlays
        handle_overlays(driver)

        wait = WebDriverWait(driver, 10)

        # Find and fill the username field
        try:
            username_field = wait.until(EC.presence_of_element_located((By.NAME, "_username")))
            username_field.clear()
            username_field.send_keys(username)
            print("Username entered successfully")
        except Exception as e:
            print(f"Error during BoxRec search: {e}")
            try:
                driver.save_screenshot("../ressources/screens/errors/boxrec_search_error.png")
                print("Screenshot saved as ../ressources/screens/errors/boxrec_search_error.png")
            except:
                pass
            return None

        # Find and fill the password field
        try:
            password_field = driver.find_element(By.NAME, "_password")
            password_field.clear()
            password_field.send_keys(password)
            print("Password entered successfully")
        except Exception as e:
            print(f"Error during BoxRec search: {e}")
            try:
                driver.save_screenshot("../ressources/screens/errors/boxrec_search_error.png")
                print("Screenshot saved as ../ressources/screens/errors/boxrec_search_error.png")
            except:
                pass
            return None

        # Find and click the login button
        try:
            login_button = driver.find_element(By.XPATH, "//button[@type='submit']")
            login_button.click()
            print("Login button clicked")
        except Exception as e:
            print(f"Error during BoxRec search: {e}")
            try:
                driver.save_screenshot("../ressources/screens/errors/boxrec_search_error.png")
                print("Screenshot saved as ../ressources/screens/errors/boxrec_search_error.png")
            except:
                pass
            return None

        # Wait for login to complete and check for success
        time.sleep(5)

        # Check if login was successful by looking for logout link or user-specific elements
        try:
            success_indicators = [
                "//a[contains(@href, 'logout')]",
                "//a[contains(text(), 'Logout')]",
                "//*[@class='navbar']",
                "//*[@id='search-bar-container']"
            ]

            for indicator in success_indicators:
                try:
                    element = WebDriverWait(driver, 3).until(
                        EC.presence_of_element_located((By.XPATH, indicator))
                    )
                    print(f"Login successful - found indicator: {indicator}")
                    return True
                except Exception:
                    continue

            print("Login may have failed - no success indicators found")
            return False

        except Exception as e:
            print(f"Error during BoxRec search: {e}")
            try:
                driver.save_screenshot("../ressources/screens/errors/boxrec_search_error.png")
                print("Screenshot saved as ../ressources/screens/errors/boxrec_search_error.png")
            except:
                pass
            return None

    except Exception as e:
        print(f"Error during BoxRec search: {e}")
        try:
            driver.save_screenshot("../ressources/screens/errors/boxrec_search_error.png")
            print("Screenshot saved as ../ressources/screens/errors/boxrec_search_error.png")
        except:
            pass
        return None

def handle_overlays(driver):
    """Handle common overlays that might block interactions"""
    try:
        # First, try to click closing links with improved error handling
        try:
            click_closing_links(driver)
        except Exception as e:
            print(f"Error in click_closing_links: {str(e)[:100]}...")
        
        # Small delay to let any overlay animations complete
        time.sleep(1)
        
        # Check for cookie consent banners and other overlays
        overlay_selectors = [
            "#onetrust-accept-btn-handler",
            ".onetrust-close-btn-handler",
            "[id*='cookie'] button",
            "[class*='cookie'] button",
            "button[aria-label*='accept']",
            "button[aria-label*='close']",
            ".modal-close",
            ".close-button",
            "[data-dismiss='modal']",
            ".overlay-close"
        ]
        
        for selector in overlay_selectors:
            try:
                elements = driver.find_elements(By.CSS_SELECTOR, selector)
                for element in elements:
                    if element.is_displayed() and element.is_enabled():
                        element.click()
                        print(f"Closed overlay using selector: {selector}")
                        time.sleep(1)
                        break
            except Exception as e:
                continue
                
        # Try pressing Escape key as a last resort
        try:
            from selenium.webdriver.common.keys import Keys
            driver.find_element(By.TAG_NAME, "body").send_keys(Keys.ESCAPE)
            print("Sent Escape key to close any remaining overlays")
            time.sleep(1)
        except:
            pass
                
    except Exception as e:
        print(f"Error handling overlays: {str(e)[:100]}...")

def search_boxrec(driver, name):
    try:
        # Go to search page (not login page since we're already logged in)
        driver.get("https://boxrec.com/en")
        time.sleep(3)  # Increased wait time for page load
        
        # Handle potential overlays first
        try:
            handle_overlays(driver)
        except Exception as e:
            print(f"Error handling overlays: {str(e)[:100]}...")
        
        time.sleep(2)  # Additional wait after overlay handling
        
        wait = WebDriverWait(driver, 20)  # Increased timeout
        
        # Multiple attempts to find and interact with search box
        search_box = None
        for attempt in range(3):
            try:
                # Wait for the search box to be both present and clickable
                search_box = wait.until(EC.element_to_be_clickable((By.ID, "si_search_text")))
                print(f"Found search box on attempt {attempt + 1}")
                break
            except Exception as e:
                print(f"Attempt {attempt + 1} failed to find search box: {str(e)[:100]}...")
                if attempt < 2:
                    time.sleep(2)
                    # Try refreshing overlays
                    try:
                        handle_overlays(driver)
                    except:
                        pass
        
        if not search_box:
            print("Failed to find search box after 3 attempts")
            return None
        
        # Scroll to the element to ensure it's in view
        driver.execute_script("arguments[0].scrollIntoView(true);", search_box)
        time.sleep(1)
        
        # Try to clear and type in the search box with multiple methods
        search_entered = False
        
        # Method 1: Direct interaction
        try:
            search_box.clear()
            search_box.send_keys(name)
            search_entered = True
            print(f"Successfully entered '{name}' in search box using direct method")
        except ElementNotInteractableException:
            print("Direct interaction failed, trying JavaScript approach")
        
        # Method 2: JavaScript approach if direct failed
        if not search_entered:
            try:
                driver.execute_script("arguments[0].value = '';", search_box)
                driver.execute_script("arguments[0].value = arguments[1];", search_box, name)
                driver.execute_script("arguments[0].dispatchEvent(new Event('input', { bubbles: true }));", search_box)
                search_entered = True
                print(f"Successfully entered '{name}' using JavaScript")
            except Exception as e:
                print(f"JavaScript method failed: {e}")
        
        if not search_entered:
            print("Failed to enter search term")
            return None
        
        # Try to select the "fighters" radio button
        try:
            fighters_radio = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "input[value='fighters']")))
            fighters_radio.click()
            print("Selected fighters filter")
        except Exception as e:
            print(f"Could not select fighters filter: {str(e)[:50]}... - continuing without it")
        
        # Submit the search - try multiple methods
        search_submitted = False
        
        # Method 1: Press Enter
        try:
            search_box.send_keys(Keys.RETURN)
            search_submitted = True
            print("Submitted search with Enter key")
        except Exception as e:
            print(f"Enter key submission failed: {str(e)[:50]}...")
        
        # Method 2: Find and click search button if Enter didn't work
        if not search_submitted:
            try:
                search_button = driver.find_element(By.CSS_SELECTOR, "input[type='submit'], button[type='submit'], .search-button")
                search_button.click()
                search_submitted = True
                print("Submitted search with button click")
            except Exception as e:
                print(f"Button submission failed: {str(e)[:50]}...")
        
        # Method 3: JavaScript form submission
        if not search_submitted:
            try:
                driver.execute_script("document.querySelector('form').submit();")
                search_submitted = True
                print("Submitted search with JavaScript")
            except Exception as e:
                print(f"JavaScript submission failed: {str(e)[:50]}...")
        
        if not search_submitted:
            print("All search submission methods failed")
            return None
        
        # Wait for results to load
        time.sleep(5)
        
        return driver.page_source      
    except Exception as e:
        print(f"Error during BoxRec search: {str(e)[:100]}...")
        # Take screenshot for debugging
        try:
            driver.save_screenshot("../ressources/screens/errors/boxrec_search_error.png")
            print("Screenshot saved as ../ressources/screens/errors/boxrec_search_error.png")
        except:
            pass
        return None

def get_first_profile_url(page_source):
    try:
        soup = BeautifulSoup(page_source, 'html.parser')
        
        # Try different selectors for profile links
        link_selectors = [
            'a[href^="/en/box-pro/"]',
            'a[href*="/box-pro/"]',
            'a[href*="/boxer/"]'
        ]
        
        for selector in link_selectors:
            link = soup.select_one(selector)
            if link:
                href = link.get('href')
                if href:
                    if href.startswith('/'):
                        return "https://boxrec.com" + href
                    else:
                        return href
        
        print("No profile link found in search results")
        return None
    except Exception as e:
        print(f"Error during BoxRec search: {e}")
        # Take screenshot for debugging
        try:
            driver.save_screenshot("../ressources/screens/errors/boxrec_search_error.png")
            print("Screenshot saved as ../ressources/screens/errors/boxrec_search_error.png")
        except:
            pass
        return None

def get_profile_html(driver, profile_url):
    try:
        driver.get(profile_url)
        time.sleep(3)  # Wait for page to load
        return driver.page_source
    except Exception as e:
        print(f"Error during BoxRec search: {e}")
        # Take screenshot for debugging
        try:
            driver.save_screenshot("../ressources/screens/errors/boxrec_search_error.png")
            print("Screenshot saved as ../ressources/screens/errors/boxrec_search_error.png")
        except:
            pass
        return None

def extract_fight_history(soup, nb_fight: int):
    """Extract fight history from BoxRec profile, limited to nb_fight fights"""
    fights = []
    
    try:
        # Find the career table div
        career_table_div = soup.find('div', class_='overflowScroll careerTable')
        if not career_table_div:
            print("No career table found")
            return fights
        
        # Find the main table
        table = career_table_div.find('table', class_='dataTable')
        if not table:
            print("No data table found in career section")
            return fights
        
        # Find all fight rows (tbody elements with id starting with 'bId')
        tbody_elements = table.find_all('tbody', id=lambda x: x and x.startswith('bId'))
        
        # Apply the limit of wanted fights
        tbody_elements = tbody_elements[:nb_fight]
        
        for tbody in tbody_elements:
            fight_row = tbody.find('tr')
            if not fight_row:
                continue
                
            fight_data = {}
            cells = fight_row.find_all('td')
            
            if len(cells) >= 10:  # Ensure we have enough cells
                try:
                    # Date (cell 1)
                    date_cell = cells[1]
                    date_link = date_cell.find('a')
                    if date_link:
                        fight_data['date'] = date_link.get_text(strip=True)
                    
                    # Boxer weight (cell 2)
                    fight_data['boxer_weight'] = cells[2].get_text(strip=True)
                    
                    # Opponent name (cell 4)
                    opponent_cell = cells[4]
                    opponent_link = opponent_cell.find('a', class_='personLink')
                    if opponent_link:
                        fight_data['opponent'] = opponent_link.get_text(strip=True)
                        fight_data['opponent_url'] = "https://boxrec.com" + opponent_link.get('href', '')
                    
                    # Opponent weight (cell 5)
                    fight_data['opponent_weight'] = cells[5].get_text(strip=True)
                    
                    # Opponent record (cell 6)
                    record_cell = cells[6]
                    wins_span = record_cell.find('span', class_='textWon')
                    losses_span = record_cell.find('span', class_='textLost')
                    draws_span = record_cell.find('span', class_='textDraw')
                    
                    if wins_span and losses_span and draws_span:
                        fight_data['opponent_record'] = f"{wins_span.get_text()}-{losses_span.get_text()}-{draws_span.get_text()}"
                    
                    # Location (cell 8)
                    location_cell = cells[8]
                    # Remove flag icons and get clean location text
                    for flag in location_cell.find_all('span', class_='flag-icon'):
                        flag.decompose()
                    fight_data['location'] = location_cell.get_text(strip=True)
                    
                    # Result (cell 9)
                    result_cell = cells[9]
                    result_div = result_cell.find('div', class_='boutResult')
                    if result_div:
                        fight_data['result'] = result_div.get_text(strip=True)
                    
                    # Rounds (cell 10)
                    fight_data['rounds'] = cells[10].get_text(strip=True)
                    
                    # Star rating (cell 11)
                    if len(cells) > 11:
                        star_cell = cells[11]
                        filled_stars = len(star_cell.find_all('i', class_='fas fa-star'))
                        fight_data['star_rating'] = filled_stars
                    
                    # Get additional details from second row if exists
                    second_row = tbody.find('tr', class_='SR')
                    if second_row:
                        second_cell = second_row.find('td', class_='secondRow')
                        if second_cell:
                            # Extract referee and judges info
                            ref_text = second_cell.get_text()
                            if 'ref:' in ref_text:
                                fight_data['details'] = ref_text.strip()
                            
                            # Extract titles if any
                            title_divs = second_cell.find_all('div', class_='titleColor')
                            if title_divs:
                                titles = []
                                for title_div in title_divs:
                                    title_link = title_div.find('a', class_='titleLink')
                                    if title_link:
                                        titles.append(title_link.get_text(strip=True))
                                fight_data['titles'] = titles
                    
                    fights.append(fight_data)
                    
                except Exception as e:
                    print(f"Error parsing fight row: {e}")
                    continue
        
        if len(fights) == 0:
            print("No fights found in career history")            
        else:
            print(f"Extracted {len(fights)} fights from career history")
        return fights
        
    except Exception as e:
        print(f"Error extracting fight history: {e}")
        return fights

def extract_boxrec_data(soup):
    """Extract boxer data from BoxRec profile soup"""
    data = {}
    
    try:
        # Extract name from h1 tag
        name_h1 = soup.find('h1')
        if name_h1:
            # Clean the name by removing icons and extra text
            name_text = name_h1.get_text(strip=True)
            # data['name'] = name_text
        
        # Extract data from profile tables with class rowTable
        tables = soup.find_all('table', class_='rowTable')
        for table in tables:
            rows = table.find_all('tr')
            for row in rows:
                cells = row.find_all(['th', 'td'])
                if len(cells) >= 2:
                    label = cells[0].get_text(strip=True).lower()
                    value = cells[1].get_text(strip=True)
                    
                    # Map common fields based on the labels
                    if 'birth name' in label:
                        data['Birth_Name'] = value
                    # elif 'height' in label:
                    #     data['height'] = value
                    # elif 'reach' in label:
                    #     data['reach'] = value
                    # elif 'stance' in label:
                    #     data['stance'] = value
                    # elif 'bouts' in label:
                    #     data['total_fights'] = value
                    elif 'debut' in label:
                        data['Debut_Career'] = value
                    elif 'division' in label:
                        data['Weight_Class'] = value
                    elif 'age' in label:
                        data['Age'] = value
                    elif 'nationality' in label:
                        data['Nationality'] = value
                    elif 'residence' in label:
                        data['Residence'] = value
                    elif 'club/gym' in label:
                        data['Training_Club'] = value
                    elif 'id#' in label:
                        data['Boxrec_Id'] = value
        
        # Extract fight history
        fight_history = extract_fight_history(soup, nb_fight=10)
        if fight_history:
            data['fight_history'] = fight_history
            data['total_professional_fights'] = len(fight_history)
        
        return data
        
    except Exception as e:
        print(f"Error extracting BoxRec data: {e}")
        return {}

def scrape_boxer_info_boxrec_selenium(
                                name,
                                username=BOXREC_USERNAME,
                                password=BOXREC_PASSWORD, 
                                show_browser=True  # Added parameter
                                ):
    """Scrape boxer info from BoxRec with login integration"""
    driver = create_selenium_driver(show_browser=show_browser)
    if not driver:
        print("Failed to create driver")
        return None
    
    try:
        # Step 1: Login to BoxRec
        login_success = login_boxrec(driver, username, password)
        if not login_success:
            print("Failed to login to BoxRec - continuing without login")
            # Continue anyway as some data might be available without login
        
        # Step 2: Search for the boxer
        print(f"Searching BoxRec for: {name}")
        search_html = search_boxrec(driver, name)
        if not search_html:
            print(f"Failed to search for {name}")
            return None
            
        # Step 3: Get profile URL
        profile_url = get_first_profile_url(search_html)
        if not profile_url:
            print(f"No profile found for {name}")
            return None
            
        print(f"Found profile URL: {profile_url}")
        
        # Step 4: Get profile data
        profile_html = get_profile_html(driver, profile_url)
        if not profile_html:
            print(f"Failed to load profile for {name}")
            return None
            
        profile_soup = BeautifulSoup(profile_html, 'html.parser')
        
        # Step 5: Extract structured data
        boxer_data = extract_boxrec_data(profile_soup)
        boxer_data['source'] = 'BoxRec'
        boxer_data['profile_url'] = profile_url
        boxer_data['login_used'] = login_success
        
        print(f"Successfully extracted data for {name}")
        return boxer_data
        
    except Exception as e:
        print(f"Error scraping {name}: {e}")
        return None
    finally:
        # Don't quit immediately - let user see what happened
        print("Keeping browser open for 5 seconds for inspection...")
        time.sleep(5)
        if driver:
            driver.quit()

# Fonction pour scraper plusieurs boxeurs avec login
def scrape_multiple_boxers_boxrec(boxer_names, max_boxers=None, show_browser=True):  # Changed defaults
    """Scrape multiple boxers from BoxRec with persistent login"""
    if max_boxers:
        boxer_names = list(boxer_names)[:max_boxers]
    
    results = []
    driver = create_selenium_driver(show_browser=show_browser)
    
    if not driver:
        print("Failed to create driver")
        return results
    
    try:
        # Login once at the beginning
        login_success = login_boxrec(driver, BOXREC_USERNAME, BOXREC_PASSWORD)
        
        # if already logged in, this will just confirm the session
        if login_success:
            print("Successfully logged in to BoxRec")
        else:
            print("Failed to login - continuing anyway")
        
        total_boxers = len(boxer_names)
        print(f"Starting to scrape {total_boxers} boxers from BoxRec...")
        
        for i, name in enumerate(boxer_names, 1):
            print(f"\n[{i}/{total_boxers}] Processing: {name}")
            
            try:
                # Search for the boxer (reusing the same driver session)
                search_html = search_boxrec(driver, name)
                if not search_html:
                    print(f"Failed to search for {name}")
                    continue
                    
                profile_url = get_first_profile_url(search_html)
                if not profile_url:
                    print(f"No profile found for {name}")
                    continue
                    
                print(f"Found profile URL: {profile_url}")
                
                profile_html = get_profile_html(driver, profile_url)
                if not profile_html:
                    print(f"Failed to load profile for {name}")
                    continue
                    
                profile_soup = BeautifulSoup(profile_html, 'html.parser')
                boxer_data = extract_boxrec_data(profile_soup)
                # boxer_data['source'] = 'BoxRec'
                # boxer_data['profile_url'] = profile_url
                # boxer_data['login_used'] = login_success
                
                boxer_df = pd.DataFrame([boxer_data])
                boxer_df['name'] = name
                # boxer_df['source'] = 'BoxRec'
                boxer_df['boxrec_profile_url'] = profile_url
                boxer_df['boxrec_login_used'] = login_success
                
                dataset_dataframe = pd.concat([dataset_dataframe, boxer_df], ignore_index=True)
                
                results.append(boxer_data)
                print(f"Successfully extracted data for {name}")
                
                # Small delay between requests
                time.sleep(2)
                
            except Exception as e:
                print(f"Error processing {name}: {e}")
                continue                
    
    finally:
        print("Scraping completed. Keeping browser open for final inspection...")
        time.sleep(5)
        if driver:
            driver.quit()
    
    print(f"\nBoxRec scraping completed! Successfully scraped {len(results)} out of {len(boxer_names)} boxers")
    return results

## REMARQUES OPTIMISATION WEB SCRAPING
- clicker sur les balises 'a' qui ont un id contenant la chaine 'closingLink' (notamment sur l'URL 'https://boxrec.com/en')
- vérifier ce qu'il fait avant de chercher un nom dans la barre de recherche

In [6]:
# RUN
boxer_names = dataset_dataframe['Name'].tolist()  # Assuming 'Name' column contains boxer names
scraped_data = scrape_multiple_boxers_boxrec(
    boxer_names,
    max_boxers=3, 
    show_browser=True  # Explicitly show browser
)

Creating Chrome driver...
Chrome driver created successfully - browser window should be visible
Attempting to login to BoxRec...
Chrome driver created successfully - browser window should be visible
Attempting to login to BoxRec...
No closing links found
No closing links found
Username entered successfully
Password entered successfully
Login button clicked
Username entered successfully
Password entered successfully
Login button clicked
Login successful - found indicator: //a[contains(@href, 'logout')]
Successfully logged in to BoxRec
Starting to scrape 3 boxers from BoxRec...

[1/3] Processing: Andy Ruiz Jr.
Login successful - found indicator: //a[contains(@href, 'logout')]
Successfully logged in to BoxRec
Starting to scrape 3 boxers from BoxRec...

[1/3] Processing: Andy Ruiz Jr.
Error clicking closing link: Message: stale element reference: stale element not found in the current frame
  (Session info: chrome=138.0.7204.101); For documentation on this error, please visit: https://www.

KeyboardInterrupt: 

## Save data

In [ ]:
if scraped_data:
    dataset_dataframe.to_csv("../data/2_boxers_cdataset.csv", index=False)
    print("Scraped data saved to ../data/2_boxers_cdataset.csv")
else:
    print("No data scraped.")

Scraped data saved to ../data/boxers_boxrec_scraped.csv
